In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from shapely import wkt

In [ ]:
'''
upload your rent dataset from dewey data, which is filtered by filter_zip_rental_for_jacob.py as consolidated
'''

rent = gpd.read_file("/Users/jacobkm/Desktop/Spring 2026/BER EA/Data_Workflow/data/sf_rentals_geo_enriched.csv")
rent_loc = gpd.GeoDataFrame(rent, geometry = gpd.points_from_xy(rent['LATITUDE'], rent['LONGITUDE']), crs = "EPSG:4326")

In [2]:
'''
upload your census tracts, zoning region, zip code tabulation region geometries as csvs and
convert these into appropriate geodataframes.
'''
zoning_bound = pd.read_csv('')
census2010 = pd.read_csv('')
census2020 = pd.read_csv('')
bound_zip = pd.read_csv('')
bound_zone = gpd.GeoDataFrame(zoningbound, geometry = zoningbound['the_geom'].apply(wkt.loads), crs = "EPSG:4326")
bound_census2010 = gpd.GeoDataFrame(census2010, geometry = census2010['the_geom'].apply(wkt.loads), crs = "EPSG:4326")
bound_census2020 = gpd.GeoDataFrame(census2020, geometry = census2020['the_geom'].apply(wkt.loads), crs = "EPSG:4326")
bound_zip = gpd.GeoDataFrame(zipbound, geometry = zipbound['geometry'].apply(wkt.loads), crs = "EPSG:4326")

NameError: name 'zoningbound' is not defined

In [ ]:
projects = pd.read_csv("/Users/jacobkm/Desktop/Spring 2026/BER EA/Data_Workflow/data/sf_housing_pipeline/Mayor's_Office_of_Housing_and_Community_Development_Affordable_Housing_Pipeline_20260412.csv")

In [ ]:
class RingAnalysis:
    """
    Encapsulates the inner/outer ring difference-in-differences analysis
    for a single construction project.
    """
    def __init__(self, center_lat, center_lon, rent_loc, public_announcement):
        self.center_lat = center_lat
        self.center_lon = center_lon
        self.center_point = gpd.GeoSeries([Point(center_lon, center_lat)], crs="EPSG:4326")
        # Reproject to a meter-based CRS for buffering in meters
        self.center_point_m = self.center_point.to_crs("EPSG:3857")
        self.rent_loc = rent_loc.to_crs("EPSG:3857")  # reproject rental data too
        self.public_announcement = pd.Timestamp(public_announcement)
        self.gdf_rings = None
        self.inclusion = None

    def distance_scan(self, innermost, outermost, N):
        """
        Creates all valid (inner < outer) ring pairs from linspace of distances (in meters).
        
        Bug fixes from original:
        - enumerate() was missing on the outer loop
        - inner_geom/outer_geom variable naming was inconsistent
        - indentation was off on inner for loop
        """
        distances = np.linspace(innermost, outermost, N, endpoint=True)
        rings = []
        for j, dist_in in enumerate(distances):               # enumerate was missing
            inner_geom = self.center_point_m.buffer(dist_in).iloc[0]
            for i, dist_out in enumerate(distances):
                if i > j:                                      # strict > so inner != outer
                    outer_geom = self.center_point_m.buffer(dist_out).iloc[0]
                    donut_geom = outer_geom.difference(inner_geom)  # naming was inconsistent
                    rings.append({
                        'inner_geom': inner_geom,
                        'outer_geom': donut_geom,
                        'ring_label': f"({dist_in:.0f},{dist_out:.0f})m"
                    })
        self.gdf_rings = gpd.GeoDataFrame(rings, geometry='outer_geom', crs="EPSG:3857")
        return self.gdf_rings

    #run this first, then use zone_encoding_data frame to manipulate this dictionary
    def ring_contained_in_zone(self, bound_zone, bound_census2010, bound_census2020, bound_zip):
    """
    For each ring pair and each boundary layer, encodes:
        0 = inner ring not contained in any zone geometry
        1 = inner ring contained, but outer donut is not fully contained in the same zone
        2 = both inner and outer fully contained in the same zone geometry

    Uses vectorized sjoin for efficiency over many rings.
    Returns a dict of 1D arrays of shape (n_rings,), one per boundary layer.
    """
    assert self.gdf_rings is not None, "Run distance_scan() first."

    boundaries = {
        'zone':       bound_zone.to_crs("EPSG:3857"),
        'census2010': bound_census2010.to_crs("EPSG:3857"),
        'census2020': bound_census2020.to_crs("EPSG:3857"),
        'zip':        bound_zip.to_crs("EPSG:3857")
    }

    # Build two GeoDataFrames from gdf_rings: one for inner geometries, one for outer
    # sjoin requires the left GDF's active geometry to be what we're testing
    rings_indexed = self.gdf_rings.reset_index(drop=True).copy()
    rings_indexed['ring_idx'] = rings_indexed.index

    inner_gdf = rings_indexed[['ring_idx', 'inner_geom']].set_geometry('inner_geom')
    outer_gdf = rings_indexed[['ring_idx', 'outer_geom']].set_geometry('outer_geom')

    results = {}

    for layer_name, boundary in boundaries.items():
        encoding = np.zeros(len(rings_indexed), dtype=np.int8)

        # Step 1: find which rings have their inner geometry contained in a zone
        # sjoin returns a row for every (ring, zone) pair that satisfies the predicate
        inner_joined = gpd.sjoin(
            inner_gdf,
            boundary[['geometry']].reset_index(drop=True).rename_geometry('geometry'),
            how='inner',
            predicate='within'         # inner_geom within zone polygon
        )
        # inner_joined has ring_idx and index_right (the zone polygon index)
        # keep only first match per ring to enforce single-zone requirement
        inner_matched = (inner_joined
                         .reset_index()
                         .drop_duplicates(subset='ring_idx', keep='first')
                         [['ring_idx', 'index_right']]
                         .rename(columns={'index_right': 'zone_idx'}))

        if inner_matched.empty:
            results[layer_name] = encoding
            continue

        # Mark all inner-contained rings as 1 initially
        encoding[inner_matched['ring_idx'].values] = 1

        # Step 2: among those, check if outer donut is also within the SAME zone polygon
        # Subset outer rings to only those whose inner was matched
        outer_candidates = outer_gdf[outer_gdf['ring_idx'].isin(inner_matched['ring_idx'])]

        outer_joined = gpd.sjoin(
            outer_candidates,
            boundary[['geometry']].reset_index(drop=True).rename_geometry('geometry'),
            how='inner',
            predicate='within'
        )
        outer_matched = (outer_joined
                         .reset_index()
                         .drop_duplicates(subset='ring_idx', keep='first')
                         [['ring_idx', 'index_right']]
                         .rename(columns={'index_right': 'zone_idx'}))

        # Step 3: upgrade to 2 only where inner and outer matched the SAME zone polygon
        both = inner_matched.merge(outer_matched, on='ring_idx', suffixes=('_inner', '_outer'))
        same_zone = both[both['zone_idx_inner'] == both['zone_idx_outer']]
        encoding[same_zone['ring_idx'].values] = 2

        results[layer_name] = encoding

    self.zone_encoding = results
    return results

    def zone_encoding_dataframe(self):
    """
    Converts zone_encoding dict into a readable DataFrame indexed by ring_label.
    Useful for filtering viable ring pairs.

    
    After rings are zone-encoded, you run this to get a dataframe which 
    can be used easily by filtering to see what rings are in each boundary.
    """
    assert self.zone_encoding is not None, "Run ring_contained_in_zone() first."
    df = pd.DataFrame(self.zone_encoding, index=self.gdf_rings['ring_label'])
    df.index.name = 'ring_label'
    return df


    def split_by_shock(self):
        """
        Splits rental data into before/after the public announcement.
        Assumes rent_loc is sorted by DATE_POSTED.
        """
        rent_sorted = self.rent_loc.sort_values('DATE_POSTED').reset_index(drop=True)
        for i in range(len(rent_sorted)):
            if rent_sorted.iloc[i]['DATE_POSTED'] > self.public_announcement:
                before = rent_sorted.iloc[:i]
                after = rent_sorted.iloc[i:]
                return before, after
        return rent_sorted, None  # no split point found

    def within_spatial_matrix(self):
        """
        Returns a boolean matrix of shape (n_rentals, n_rings).
        inclusion[i, j] = 1 if rental i falls within ring pair j's outer donut geometry.

        Bug fixes from original:
        - 'for i in rent_loc' doesn't give row index; use iterrows() or vectorized sjoin
        - 'A = True' is an assignment, should be '== True' or just 'if A'
        - inclusion_space was never defined, used inclusion instead
        
        Note: vectorized spatial join is far faster than a Python loop over 1M rows.
        """
        assert self.gdf_rings is not None, "Run distance_scan() first."

        rent_indexed = self.rent_loc.reset_index(drop=True)
        rent_indexed['rental_idx'] = rent_indexed.index
        inclusion = np.zeros((len(rent_indexed), len(self.gdf_rings)), dtype=np.int8)

        # Vectorized: spatial join is orders of magnitude faster than looping over 1M rows
        rings_indexed = self.gdf_rings.reset_index(drop=True)
        rings_indexed['ring_idx'] = rings_indexed.index

        joined = gpd.sjoin(rent_indexed[['rental_idx', 'geometry']],
                           rings_indexed[['ring_idx', 'outer_geom']].set_geometry('outer_geom'),
                           how='inner', predicate='within')

        for row in joined.itertuples():
            inclusion[row.rental_idx, row.ring_idx] = 1

        self.inclusion = inclusion
        return inclusion

    def count_per_ring(self):
        """
        Returns a DataFrame with the count of rentals per ring pair.

        Bug fixes from original:
        - 'for i, ring in gdf_rings' needs iterrows()
        - dict key used ring['...'] which fails without .iterrows()
        """
        assert self.inclusion is not None, "Run within_spatial_matrix() first."
        counts = []
        for i, ring in self.gdf_rings.iterrows():
            counts.append({
                'ring_label': ring['ring_label'],
                'count': int(np.sum(self.inclusion[:, i]))
            })
        return pd.DataFrame(counts)
    """
    Among best choice of rings, via number and inclusion within zones before and after the announcment
    we take the closest rental units and see at what time date the price ceases to change a lot
    
    """